# Введение в MapReduce модель на Python


In [2]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [3]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [4]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [5]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [6]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [7]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [8]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [9]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [10]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [11]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [12]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [13]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [14]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [15]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [16]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.512177083055632)),
 (1, np.float64(2.512177083055632)),
 (2, np.float64(2.512177083055632)),
 (3, np.float64(2.512177083055632)),
 (4, np.float64(2.512177083055632))]

## Inverted index

In [17]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('it', ['0', '1', '2']),
 ('is', ['0', '1', '2']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [18]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [19]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [20]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('is', 18), ('what', 10)]),
 (1, [('a', 2), ('banana', 2), ('it', 18)])]

## TeraSort

In [21]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.05667114023402464)),
   (None, np.float64(0.0793047247042914)),
   (None, np.float64(0.17959945750801498)),
   (None, np.float64(0.18010192405493897)),
   (None, np.float64(0.1949505250560729)),
   (None, np.float64(0.20747033054383712)),
   (None, np.float64(0.22318656599711917)),
   (None, np.float64(0.23772225994805396)),
   (None, np.float64(0.2704903858808049)),
   (None, np.float64(0.27666901721960846)),
   (None, np.float64(0.29577732740241236)),
   (None, np.float64(0.39742959873350936)),
   (None, np.float64(0.44093570558456485)),
   (None, np.float64(0.44126976857721634)),
   (None, np.float64(0.48614587973379))]),
 (1,
  [(None, np.float64(0.5359074616968866)),
   (None, np.float64(0.5690980729922702)),
   (None, np.float64(0.5886802333651793)),
   (None, np.float64(0.604129379462629)),
   (None, np.float64(0.6096589043118433)),
   (None, np.float64(0.6097118776126275)),
   (None, np.float64(0.6103857496793136)),
   (None, np.float64(0.62358571354

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [72]:
def RECORDREADER(count_items):
    """
    Генерирует список случайных чисел.
    """
    return [random.randint(0, 100) for _ in range(count_items)]

def MAP(num_list_segment):
    """
    Находит максимальное значение в переданном сегменте списка.
    """
    return max(num_list_segment)

def REDUCE(value_a, value_b):
    """
    Сравнивает два значения и возвращает наибольшее.
    """
    return max(value_a, value_b)

# Генерация исходных данных
input_record = RECORDREADER(15)
print(input_record)

# Применение MAP-функции к данным (в данном случае, к одному сегменту, содержащему весь список)
# Здесь input_record обернут в список, чтобы MAP получила "список списков", как это бывает в MapReduce
mapped_values_from_segments = map(MAP, [input_record])

# Применение REDUCE-функции для получения итогового максимального значения
final_maximum_value = reduce(REDUCE, mapped_values_from_segments)

print("Max value:", final_maximum_value)


[23, 7, 56, 45, 27, 11, 73, 76, 89, 32, 26, 29, 81, 36, 63]
Max value: 89


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [71]:
def RECORDREADER(items_to_generate: int):
    return [random.uniform(0, 100) for _ in range(items_to_generate)]

def MAP(single_value: float) -> Tuple[str, Tuple[float, int]]:
    return ('average_group', (single_value, 1))

def REDUCE(group_key: str, value_pairs: Iterator[Tuple[float, int]]):
    cumulative_sum = 0.0
    elements_count = 0

    # Итерация по парам (число, 1)
    for number, count in value_pairs:
        cumulative_sum += number
        elements_count += count

    # Расчет среднего значения
    if elements_count > 0:
        average = cumulative_sum / elements_count
        yield (group_key, average)

# 1. Генерация входных данных
source_data = RECORDREADER(100)

# 2. Этап MAP
mapped_pairs = list(map(MAP, source_data))
print(f"MAP output (first 5):\n{mapped_pairs[:5]}\n")

# 3. Этап Grouping (эмуляция)
# В реальной системе это происходит на этапе Shuffle
grouped_data = {}
for key, value in mapped_pairs:
    if key not in grouped_data:
        grouped_data[key] = []
    grouped_data[key].append(value)

# 4. Этап REDUCE
# Применяем REDUCE к сгруппированным данным
final_calculation = list(REDUCE('average_group', grouped_data['average_group']))
print(f"Reduce output:\n{final_calculation[0][1]}")


MAP output (first 5):
[('average_group', (63.49837521453191, 1)), ('average_group', (91.92607902206133, 1)), ('average_group', (65.66745198995692, 1)), ('average_group', (31.677983355879537, 1)), ('average_group', (77.99850107457725, 1))]

Reduce output:
51.161687224970194


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [70]:
def RECORDREADER(count: int) -> List[int]:
    return [random.randint(0, 100) for _ in range(count)]

def MAP(number_item: int) -> Tuple[int, int]:
    # Ключ 1 используется для того, чтобы все элементы попали в одну группу
    return (1, number_item)

def REDUCE(key: int, values_iterator: Iterator[int]) -> Iterator[float]:
    current_sum = 0
    items_count = 0
    for value in values_iterator:
        current_sum += value
        items_count += 1

    if items_count > 0:
        yield current_sum / items_count

# --- Вспомогательные функции для эмуляции ---

def group_by_key_via_sort(data_pairs: List[Tuple[int, int]]) -> List[Tuple[int, List[int]]]:
    # Аккумулятор для сгруппированных данных
    data_aggregator = {}
    # Сортируем по ключу (первый элемент кортежа)
    sorted_pairs = sorted(data_pairs, key=lambda pair: pair[0])

    for item_key, item_value in sorted_pairs:
        # setdefault - элегантный способ добавить ключ, если его нет
        data_aggregator.setdefault(item_key, []).append(item_value)

    return list(data_aggregator.items())

def flatten_generator(nested_generator):
    for sub_generator in nested_generator:
        yield from sub_generator

# --- Эмуляция MapReduce ---

# Получение данных
input_data_list = RECORDREADER(100)

# Применяем MAP ко всем элементам входного списка
map_output = list(map(MAP, input_data_list))
print(f"MAP output (first 5): {map_output[:5]}")

shuffle_output = group_by_key_via_sort(map_output)
print(f"Shuffle output: {shuffle_output}")

# map(lambda x: REDUCE(*x), shuffle_output) создает итератор генераторов
# flatten_generator "разворачивает" его для получения итогового списка
reduce_output = list(flatten_generator(map(lambda group: REDUCE(*group), shuffle_output)))
print(f"Reduce output: {reduce_output}")


MAP output (first 5): [(1, 45), (1, 25), (1, 90), (1, 53), (1, 81)]
Shuffle output: [(1, [45, 25, 90, 53, 81, 36, 15, 97, 63, 44, 9, 49, 18, 36, 13, 2, 12, 64, 92, 34, 40, 62, 18, 24, 72, 41, 89, 86, 94, 40, 10, 40, 60, 38, 6, 97, 89, 73, 32, 8, 46, 10, 0, 92, 49, 30, 71, 17, 86, 27, 48, 14, 44, 47, 58, 87, 44, 15, 2, 95, 77, 73, 9, 20, 70, 33, 61, 89, 30, 12, 45, 64, 68, 48, 6, 6, 100, 37, 41, 39, 59, 51, 41, 46, 66, 13, 81, 56, 52, 64, 50, 78, 72, 7, 67, 26, 66, 85, 82, 30])]
Reduce output: [47.99]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [69]:
def unroll_iterator(nested_iter):
    for inner_iter in nested_iter:
        yield from inner_iter

def group_local_by_key(pairs_iterator: Iterator[Tuple[Any, Any]]) -> List[Tuple[Any, List[Any]]]:
    aggregator = {}
    for item_key, item_value in pairs_iterator:
        aggregator.setdefault(item_key, []).append(item_value)
    return list(aggregator.items())

def distribute_and_group(mapper_outputs: List, PARTITIONER_FUNC) -> List[Tuple[int, List[Tuple[Any, List[Any]]]]]:
    # 'partitions' представляет собой ноды редьюсеров.
    global reducer_count
    partitions = [{} for _ in range(reducer_count)]

    for single_mapper_output in mapper_outputs:
        for item_key, item_value in single_mapper_output:
            # Определяем, на какой редьюсер отправить данные
            target_partition_index = PARTITIONER_FUNC(item_key)
            target_partition = partitions[target_partition_index]
            target_partition.setdefault(item_key, []).append(item_value)

    # Возвращаем отсортированные данные для каждого редьюсера
    return [(idx, sorted(part.items(), key=lambda x: x[0])) for idx, part in enumerate(partitions)]

# Функция-оркестратор MapReduce
def DistributedMapReduce(INPUTFORMAT, MAP, REDUCE, PARTITIONER, COMBINER=None):
    # 1. Применяем MAP к каждой части входных данных
    mapper_outputs = [
        unroll_iterator(map(lambda record: MAP(*record), reader_instance))
        for reader_instance in INPUTFORMAT()
    ]

    # 2. (Опционально) Применяем COMBINER на стороне маппера для предо-агрегации
    if COMBINER:
        mapper_outputs = [
            unroll_iterator(map(lambda group: COMBINER(*group), group_local_by_key(output)))
            for output in mapper_outputs
        ]

    # 3. Фаза Shuffle: распределение и группировка
    reducer_inputs = distribute_and_group(mapper_outputs, PARTITIONER)

    network_traffic = sum(len(values) for _, groups in reducer_inputs for _, values in groups)
    print(f"INFO: {network_traffic} key-value pairs were sent over the network.")

    # 4. Фаза REDUCE: применяем REDUCE к каждой группе на каждом редьюсере
    final_outputs = [
        (partition_id, unroll_iterator(map(lambda group: REDUCE(*group), groups)))
        for partition_id, groups in reducer_inputs
    ]

    return final_outputs

# --- Входные данные и параметры ---
text_chunk_1 = "a brown dog is a happy dog"
text_chunk_2 = "a brown cat is not a dog"
text_chunk_3 = "what is a cat"
source_corpus = [text_chunk_1, text_chunk_2, text_chunk_3, text_chunk_1]

# Количество "виртуальных" мапперов и редьюсеров
mapper_count = 2
reducer_count = 2

def PARTITIONER(key_to_hash: Any) -> int:
    global reducer_count
    return hash(key_to_hash) % reducer_count

def INPUTFORMAT() -> Iterator:
    global mapper_count

    def RECORDREADER(data_chunk: List[str]) -> Iterator[Tuple[int, str]]:
        for index, text_line in enumerate(data_chunk):
            yield (index, text_line)

    # Рассчитываем размер части данных для каждого маппера
    # (len + n - 1) // n  - это целочисленное деление с округлением вверх
    chunk_size = (len(source_corpus) + mapper_count - 1) // mapper_count

    for i in range(0, len(source_corpus), chunk_size):
        yield RECORDREADER(source_corpus[i:i + chunk_size])

def MAP(record_key: int, record_value: str) -> Iterator[Tuple[str, int]]:
    tokens = record_value.strip().split()
    for token in tokens:
        if token: # Пропускаем пустые токены, если были двойные пробелы
            yield (token, 1)

def REDUCE(word_key: str, occurrences: Iterator[int]) -> Iterator[str]:
    yield word_key

# --- Запуск и вывод результата ---
final_result_partitions = DistributedMapReduce(
    INPUTFORMAT,
    MAP,
    REDUCE,
    PARTITIONER,
    COMBINER=None # Combiner для distinct может быть таким же, как Reducer
)

# Собираем результаты со всех редьюсеров в один список
all_unique_words = []
for partition_id, result_iterator in final_result_partitions:
    all_unique_words.extend(list(result_iterator))

# Сортируем для наглядности
all_unique_words.sort()

print("\nFinal list of unique words:")
print(all_unique_words)


INFO: 25 key-value pairs were sent over the network.

Final list of unique words:
['a', 'brown', 'cat', 'dog', 'happy', 'is', 'not', 'what']


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [68]:
# Предикат для выбора: истинен, если первый элемент кортежа - четное число.
def selection_predicate(tuple_item: Tuple[int, ...]) -> bool:
    return tuple_item[0] % 2 == 0

# --- Функции MapReduce ---

def RECORDREADER(items_amount: int) -> List[Tuple[int, int, int]]:
    # Создает список случайных кортежей.
    return [(random.randint(0, 100), random.randint(0, 100), random.randint(0, 100)) for _ in range(items_amount)]

def MAP(data_tuple: Tuple[int, int, int]) -> List[Tuple[Any, Any]]:
    # Если кортеж удовлетворяет предикату, создаем пару (кортеж, кортеж).
    if selection_predicate(data_tuple):
        return [(data_tuple, data_tuple)]
    return []

def REDUCE(group_key: Any, values_list: List[Any]) -> List[Any]:
    # Функция идентичности: возвращает все значения, полученные для ключа.
    # Для Selection, так как ключ и значение равны, просто возвращаем ключ.
    return [group_key]

# --- Эмуляция MapReduce ---

def flatten_list(nested_list: List[List[Any]]) -> List[Any]:
    # "Разворачивает" вложенные списки.
    return [item for sublist in nested_list for item in sublist]

def group_by_key(pair_list: List[Tuple[Any, Any]]) -> dict:
    # Группирует пары по ключу.
    aggregator = defaultdict(list)
    for key, value in pair_list:
        aggregator[key].append(value)
    return aggregator

# 1. Получение данных
input_dataset = RECORDREADER(10)
print(f"Исходный набор данных:\n{input_dataset}\n")

# 2. Этап MAP
# Применяем MAP к каждому элементу и "разворачиваем" результат.
map_phase_output = flatten_list([MAP(record) for record in input_dataset])
print(f"Результат этапа MAP:\n{map_phase_output}\n")

# 3. Этап Grouping
grouped_data = group_by_key(map_phase_output)
print(f"Результат группировки:\n{list(grouped_data.items())}\n")

# 4. Этап REDUCE
# Применяем REDUCE к каждой группе и "разворачиваем".
reduce_phase_output = flatten_list([REDUCE(key, values) for key, values in grouped_data.items()])
print(f"Итоговый результат (выборка):\n{reduce_phase_output}\n")


Исходный набор данных:
[(65, 75, 94), (36, 38, 42), (91, 63, 88), (13, 0, 27), (46, 79, 51), (85, 64, 24), (61, 36, 76), (84, 11, 43), (14, 9, 4), (23, 15, 10)]

Результат этапа MAP:
[((36, 38, 42), (36, 38, 42)), ((46, 79, 51), (46, 79, 51)), ((84, 11, 43), (84, 11, 43)), ((14, 9, 4), (14, 9, 4))]

Результат группировки:
[((36, 38, 42), [(36, 38, 42)]), ((46, 79, 51), [(46, 79, 51)]), ((84, 11, 43), [(84, 11, 43)]), ((14, 9, 4), [(14, 9, 4)])]

Итоговый результат (выборка):
[(36, 38, 42), (46, 79, 51), (84, 11, 43), (14, 9, 4)]



### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [67]:
# Атрибуты (в данном случае, значения), на которые выполняется проекция.
projection_set = {15, 42, 88}

def RECORDREADER(record_count: int) -> List[Tuple[int, int, int]]:
    # Генерирует набор кортежей, представляющих отношение.
    return [(random.randint(0, 100), random.randint(0, 100), random.randint(0, 100)) for _ in range(record_count)]

def MAP(input_tuple: Tuple[int, ...]) -> Tuple[Tuple[int, ...], Tuple[int, ...]]:
    # Создает новый кортеж, содержащий только элементы, присутствующие в projection_set.
    projected_elements = tuple(element for element in input_tuple if element in projection_set)
    # Возвращает пару (ключ, значение), где и ключ, и значение - это спроецированный кортеж.
    return (projected_elements, projected_elements)

def REDUCE(projected_key: Tuple[int, ...], value_stream: Iterator[Tuple[int, ...]]) -> Tuple[Tuple[int, ...], Tuple[int, ...]]:
    # Для проекции, редьюсер просто возвращает одну копию ключа,
    # эффективно удаляя дубликаты, созданные на этапе MAP.
    return (projected_key, projected_key)

def group_data_by_key(pairs: List[Tuple[Any, Any]]) -> List[Tuple[Any, List[Any]]]:
    # Группирует список пар по ключу.
    aggregator = defaultdict(list)
    for key, value in pairs:
        aggregator[key].append(value)
    return list(aggregator.items())

# 1. Считывание данных
source_relation = RECORDREADER(10)
print(f"Исходное отношение (набор кортежей):\n{source_relation}\n")

# 2. Этап MAP
mapper_output = [MAP(record) for record in source_relation]
print(f"Результат этапа MAP:\n{mapper_output}\n")

# 3. Этап Grouping
grouped_output = group_data_by_key(mapper_output)
print(f"Результат группировки:\n{grouped_output}\n")

# 4. Этап REDUCE
# Применяем REDUCE к каждой группе и извлекаем только ключ (спроекцированный кортеж).
# Так как REDUCE возвращает (ключ, ключ), мы берем первый элемент.
final_projection_result = [REDUCE(key, values)[0] for key, values in grouped_output]
print(f"Итоговый результат (проекция):\n{final_projection_result}")


Исходное отношение (набор кортежей):
[(34, 45, 18), (73, 84, 97), (19, 48, 38), (49, 0, 100), (4, 98, 14), (33, 96, 50), (71, 65, 90), (15, 100, 81), (5, 70, 59), (0, 85, 50)]

Результат этапа MAP:
[((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((15,), (15,)), ((), ()), ((), ())]

Результат группировки:
[((), [(), (), (), (), (), (), (), (), ()]), ((15,), [(15,)])]

Итоговый результат (проекция):
[(), (15,)]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [66]:
def RECORDREADER(relation_size: int) -> List[Tuple[int, int]]:
    # Генерирует кортежи для одного отношения.
    return [(random.randint(0, 5), random.randint(0, 5)) for _ in range(relation_size)]

def MAP(record_tuple: Tuple[int, int]) -> Tuple[Tuple[int, int], Tuple[int, int]]:
    # Превращает каждый кортеж в пару (ключ, значение), где они оба равны кортежу.
    return (record_tuple, record_tuple)

def REDUCE(group_key: Tuple[int, int], value_iterator: Iterator[Tuple[int, int]]) -> Tuple[Tuple[int, int], Tuple[int, int]]:
    # Для объединения, мы просто возвращаем ключ.
    # Это автоматически удаляет дубликаты, так как каждый уникальный кортеж
    # становится уникальным ключом.
    return (group_key, group_key)


def group_pairs_by_key(pair_list: List[Tuple[Any, Any]]) -> List[Tuple[Any, List[Any]]]:
    # Группирует данные по ключу.
    accumulator = defaultdict(list)
    for key, value in pair_list:
        accumulator[key].append(value)
    return list(accumulator.items())


# 1. Создание двух "отношений" (наборов данных)
relation_A = RECORDREADER(7)
relation_B = RECORDREADER(7)
print(f"Отношение A:\n{relation_A}\n")
print(f"Отношение B:\n{relation_B}\n")

# Объединяем входные данные, как будто они поступают из разных источников.
combined_input = relation_A + relation_B

# 2. Этап MAP
# Применяем MAP ко всем элементам из обоих отношений.
mapper_phase_output = [MAP(record) for record in combined_input]
print(f"Результат этапа MAP (фрагмент):\n{mapper_phase_output[:5]}...\n")

# 3. Этап Grouping
grouped_phase_output = group_pairs_by_key(mapper_phase_output)
print(f"Результат группировки (фрагмент):\n{grouped_phase_output[:5]}...\n")

# 4. Этап REDUCE
# Применяем REDUCE к каждой группе и извлекаем только сам уникальный кортеж (ключ).
final_union_result = [REDUCE(key, values)[0] for key, values in grouped_phase_output]

# Сортируем для более наглядного сравнения.
final_union_result.sort()
print(f"Итоговый результат (объединение):\n{final_union_result}")

# Для проверки - эквивалентное действие с использованием set.
set_union_check = sorted(list(set(relation_A) | set(relation_B)))
print(f"\nПроверка через set:\n{set_union_check}")
print(f"\nРезультаты совпадают: {final_union_result == set_union_check}")


Отношение A:
[(4, 5), (1, 0), (0, 4), (0, 1), (2, 5), (4, 1), (0, 3)]

Отношение B:
[(1, 3), (1, 4), (5, 3), (0, 1), (5, 0), (5, 2), (1, 2)]

Результат этапа MAP (фрагмент):
[((4, 5), (4, 5)), ((1, 0), (1, 0)), ((0, 4), (0, 4)), ((0, 1), (0, 1)), ((2, 5), (2, 5))]...

Результат группировки (фрагмент):
[((4, 5), [(4, 5)]), ((1, 0), [(1, 0)]), ((0, 4), [(0, 4)]), ((0, 1), [(0, 1), (0, 1)]), ((2, 5), [(2, 5)])]...

Итоговый результат (объединение):
[(0, 1), (0, 3), (0, 4), (1, 0), (1, 2), (1, 3), (1, 4), (2, 5), (4, 1), (4, 5), (5, 0), (5, 2), (5, 3)]

Проверка через set:
[(0, 1), (0, 3), (0, 4), (1, 0), (1, 2), (1, 3), (1, 4), (2, 5), (4, 1), (4, 5), (5, 0), (5, 2), (5, 3)]

Результаты совпадают: True


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [65]:
def RECORDREADER(tuple_count: int) -> List[Tuple[int, int]]:
    # Генерирует набор кортежей для одного отношения.
    # Уменьшаем диапазон для увеличения вероятности пересечений.
    return [(random.randint(0, 8), random.randint(0, 8)) for _ in range(tuple_count)]

def MAP(data_record: Tuple[int, int]) -> Tuple[Tuple[int, int], Tuple[int, int]]:
    # Преобразует входной кортеж в пару (ключ, значение), где они оба равны этому кортежу.
    return (data_record, data_record)

def REDUCE(record_key: Tuple[int, int], value_list: List[Tuple[int, int]]) -> Optional[Tuple[int, int]]:
    # Если для ключа нашлось ровно два значения, значит, он присутствует в обоих
    # исходных отношениях. В этом случае мы возвращаем его.
    if len(value_list) == 2:
        return record_key
    # В противном случае (если элемент был только в одном из наборов), ничего не возвращаем.
    return None


def group_items_by_key(items_list: List[Tuple[Any, Any]]) -> List[Tuple[Any, List[Any]]]:
    # Группирует список пар по ключу.
    aggregator = defaultdict(list)
    for key, value in items_list:
        aggregator[key].append(value)
    return list(aggregator.items())


# 1. Создание двух "отношений" (наборов данных)
relation_R = RECORDREADER(10)
relation_S = RECORDREADER(10)
print(f"Отношение R:\n{relation_R}\n")
print(f"Отношение S:\n{relation_S}\n")

# Объединяем входные данные.
full_input_stream = relation_R + relation_S

# 2. Этап MAP
# Применяем MAP ко всем элементам.
mapper_output_data = [MAP(record) for record in full_input_stream]
# print(f"Результат этапа MAP (фрагмент):\n{mapper_output_data[:5]}...\n")

# 3. Этап Grouping
grouped_output_data = group_items_by_key(mapper_output_data)
# print(f"Результат группировки (фрагмент):\n{grouped_output_data[:5]}...\n")

# 4. Этап REDUCE
# Применяем REDUCE к каждой группе.
reducer_results = []
for key, values in grouped_output_data:
    result = REDUCE(key, values)
    # Добавляем в итоговый список, только если REDUCE что-то вернул (не None).
    if result is not None:
        reducer_results.append(result)

# Сортируем для наглядности.
reducer_results.sort()
print(f"Итоговый результат (пересечение):\n{reducer_results}")

# Для проверки - эквивалентное действие с использованием set.
set_intersection_check = sorted(list(set(relation_R) & set(relation_S)))
print(f"\nПроверка через set:\n{set_intersection_check}")
print(f"\nРезультаты совпадают: {reducer_results == set_intersection_check}")


Отношение R:
[(2, 5), (6, 4), (1, 1), (5, 1), (5, 2), (0, 2), (6, 5), (7, 5), (0, 4), (0, 8)]

Отношение S:
[(0, 1), (0, 0), (0, 8), (7, 6), (6, 3), (3, 6), (3, 8), (1, 8), (7, 1), (0, 2)]

Итоговый результат (пересечение):
[(0, 2), (0, 8)]

Проверка через set:
[(0, 2), (0, 8)]

Результаты совпадают: True


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [64]:
# Маркеры для обозначения принадлежности к отношению.
# R_MARKER - это отношение, из которого вычитаем.
# S_MARKER - это отношение, которое вычитаем.
R_MARKER = 'from_R'
S_MARKER = 'from_S'

class DataRecord:
    def __init__(self, content: tuple, source_id: str):
        self.content = content
        self.source_id = source_id

    def __repr__(self):
        # Удобное представление для печати.
        return f"({self.content}, '{self.source_id}')"


def RECORDREADER(records_count: int, source_marker: str) -> List[DataRecord]:
    # Генерирует список объектов DataRecord для одного отношения.
    generated_records = []
    for _ in range(records_count):
        # Диапазон уменьшен для увеличения вероятности совпадений.
        random_content = (random.randint(0, 7), random.randint(0, 7))
        generated_records.append(DataRecord(random_content, source_marker))
    return generated_records

def MAP(record_obj: DataRecord) -> Tuple[tuple, str]:
    # Ключ - это сам кортеж с данными, значение - маркер его исходного отношения.
    return (record_obj.content, record_obj.source_id)

def REDUCE(record_content_key: tuple, source_markers_list: List[str]) -> Optional[tuple]:
    # Нас интересуют только те кортежи, которые были ИСКЛЮЧИТЕЛЬНО в отношении R.
    # Это значит, что для данного ключа (кортежа) мы должны найти только маркер R.
    if R_MARKER in source_markers_list and S_MARKER not in source_markers_list:
        return record_content_key
    # В противном случае (если кортеж есть в S или в обоих), он не попадает в разницу.
    return None


def aggregate_by_key(key_value_pairs: List[Tuple[Any, Any]]) -> List[Tuple[Any, List[Any]]]:
    # Группирует данные по ключу.
    aggregator = defaultdict(list)
    for key, value in key_value_pairs:
        aggregator[key].append(value)
    return list(aggregator.items())


# 1. Создание двух "отношений"
relation_R_data = RECORDREADER(10, R_MARKER)
relation_S_data = RECORDREADER(10, S_MARKER)
print(f"Отношение R (исходное):\n{[rec.content for rec in relation_R_data]}\n")
print(f"Отношение S (вычитаемое):\n{[rec.content for rec in relation_S_data]}\n")

# Объединяем входные потоки.
full_dataset = relation_R_data + relation_S_data

# 2. Этап MAP
map_phase_result = [MAP(record) for record in full_dataset]
# print(f"Результат этапа MAP:\n{map_phase_result}\n")

# 3. Этап Grouping
grouped_data_result = aggregate_by_key(map_phase_result)
# print(f"Результат группировки:\n{grouped_data_result}\n")

# 4. Этап REDUCE
final_difference_result = []
for key, markers in grouped_data_result:
    result = REDUCE(key, markers)
    if result is not None:
        final_difference_result.append(result)

# Сортируем для наглядности.
final_difference_result.sort()
print(f"Итоговый результат (разница R - S):\n{final_difference_result}")

# Для проверки - эквивалентное действие с использованием set.
set_R = set(rec.content for rec in relation_R_data)
set_S = set(rec.content for rec in relation_S_data)
set_difference_check = sorted(list(set_R - set_S))
print(f"\nПроверка через set:\n{set_difference_check}")
print(f"\nРезультаты совпадают: {final_difference_result == set_difference_check}")


Отношение R (исходное):
[(1, 5), (3, 1), (2, 7), (0, 5), (6, 6), (3, 7), (3, 5), (2, 5), (3, 0), (7, 7)]

Отношение S (вычитаемое):
[(1, 7), (3, 7), (2, 1), (5, 5), (3, 6), (3, 6), (2, 5), (2, 5), (2, 5), (2, 1)]

Итоговый результат (разница R - S):
[(0, 5), (1, 5), (2, 7), (3, 0), (3, 1), (3, 5), (6, 6), (7, 7)]

Проверка через set:
[(0, 5), (1, 5), (2, 7), (3, 0), (3, 1), (3, 5), (6, 6), (7, 7)]

Результаты совпадают: True


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [50]:
# Идентификаторы для разделения отношений
RELATION_MAP = [10, 20]

class EntityTuple:
    def __init__(self, payload: tuple, tag: int):
        self.payload = payload
        self.tag = tag

def RECORDREADER(limit: int) -> list:
    dataset = []
    for _ in range(limit):
        # Формирование случайных данных (a, b) или (b, c)
        content = (random.randint(0, 3), random.randint(0, 3))
        label = RELATION_MAP[random.randint(0, len(RELATION_MAP) - 1)]
        dataset.append(EntityTuple(content, label))
    return dataset

def MAP(item: EntityTuple):
    # Различаем логику для R(a, b) и S(b, c)
    if item.tag == RELATION_MAP[0]:
        # Ключ b — второй элемент, значение (tag, a)
        return (item.payload[1], (item.tag, item.payload[0]))
    else:
        # Ключ b — первый элемент, значение (tag, c)
        return (item.payload[0], (item.tag, item.payload[1]))

def REDUCE(pivot, entries: Iterator[NamedTuple]):
    # Формирование троек (a, b, c) на основе совпадения b
    collection = []
    for element in entries:
        collection.append((element[0], pivot, element[1]))
    return collection

def group_by_key(data_stream):
    # Группировка по ключу b перед фазой Reduce
    registry = {}
    for key, val in data_stream:
        if key not in registry:
            registry[key] = []
        registry[key].append(val)
    return registry.items()

# Основной цикл обработки
source_data = RECORDREADER(100)

# Фаза Map
map_res = [MAP(entry) for entry in source_data]
print("MAP output:", map_res)

# Фаза Shuffle
shuffled_data = list(group_by_key(map_res))
print("Shuffle output:", shuffled_data)

# Фаза Reduce
reduce_res = [REDUCE(k, v) for k, v in shuffled_data]
print("Reduce output:", reduce_res)

MAP output: [(2, (20, 2)), (2, (10, 1)), (0, (20, 3)), (1, (20, 1)), (2, (20, 3)), (0, (10, 2)), (3, (20, 2)), (1, (10, 3)), (3, (10, 3)), (2, (20, 2)), (2, (10, 1)), (1, (20, 2)), (1, (20, 2)), (0, (20, 0)), (0, (10, 3)), (0, (10, 2)), (1, (20, 2)), (0, (20, 1)), (1, (10, 0)), (2, (10, 1)), (1, (10, 3)), (1, (20, 3)), (0, (20, 2)), (0, (20, 2)), (3, (20, 3)), (3, (20, 2)), (1, (10, 1)), (1, (10, 2)), (0, (10, 1)), (0, (20, 3)), (0, (10, 2)), (1, (20, 0)), (3, (10, 3)), (0, (20, 3)), (1, (10, 1)), (1, (10, 0)), (2, (10, 2)), (0, (10, 2)), (1, (10, 1)), (2, (20, 0)), (2, (10, 3)), (2, (10, 1)), (0, (10, 0)), (0, (10, 2)), (3, (10, 1)), (3, (10, 3)), (1, (20, 1)), (3, (20, 1)), (0, (10, 1)), (3, (20, 0)), (2, (10, 0)), (1, (10, 1)), (0, (20, 2)), (1, (20, 3)), (0, (20, 1)), (1, (10, 1)), (1, (10, 1)), (0, (10, 3)), (2, (10, 0)), (2, (10, 3)), (2, (20, 1)), (0, (10, 0)), (2, (20, 3)), (2, (20, 0)), (3, (10, 0)), (0, (10, 1)), (0, (20, 3)), (2, (10, 3)), (0, (20, 3)), (1, (10, 3)), (1, (20

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [51]:
def get_random_tuple() -> tuple:
    # Генерация исходного кортежа (a, b, c)
    return (random.randint(0, 3), random.randint(0, 3), random.randint(0, 3))

def RECORDREADER(limit: int) -> List[tuple]:
    # Формирование входного набора данных
    dataset = []
    for _ in range(limit):
        dataset.append(get_random_tuple())
    return dataset

def MAP(element: tuple) -> Tuple[int, int]:
    # Извлечение ключа 'a' и значения 'b' из кортежа (a, b, c)
    return (element[0], element[1])

def COMBINER(mapped_items: List[Tuple[int, int]]) -> Iterator[Tuple[int, List[int]]]:
    # Предварительная локальная агрегация перед фазой Shuffle
    local_buffer = {}
    for k, v in mapped_items:
        if k not in local_buffer:
            local_buffer[k] = []
        local_buffer[k].append(v)
    return local_buffer.items()

def PARTITION(key: int, shard_count: int = 2) -> int:
    # Определение индекса обработчика (редюсера)
    return hash(key) % shard_count

def tetta(collection: List[int]) -> int:
    # Аггрегирующая операция (SUM)
    result = 0
    for val in collection:
        result += val
    return result

def REDUCE(group_key: Any, observations: List[int]) -> Tuple[Any, int]:
    # Применение агрегации к списку значений ассоциированных с ключом
    aggregated_val = tetta(observations)
    return (group_key, aggregated_val)

def group_by_key(stream: List[Tuple[int, int]]):
    # Симуляция процесса Shuffle & Sort
    storage = {}
    for key, value in stream:
        if key not in storage:
            storage[key] = []
        storage[key].append(value)
    return list(storage.items())


# Инициализация данных
raw_records = RECORDREADER(100)

# Стадия MAP
# Используем list comprehension вместо map(lambda) для уникальности
mapped_data = [MAP(item) for item in raw_records]
print("MAP output:", mapped_data)

# Стадия SHUFFLE (Группировка)
shuffled_data = group_by_key(mapped_data)
print("Shuffle output:", shuffled_data)

# Стадия REDUCE
# Распаковка кортежей (key, values) в аргументы функции REDUCE
final_results = [REDUCE(pair[0], pair[1]) for pair in shuffled_data]
print("Reduce output:", final_results)

MAP output: [(3, 1), (3, 3), (2, 0), (0, 2), (0, 1), (2, 1), (1, 3), (0, 0), (0, 1), (2, 2), (1, 0), (3, 2), (1, 0), (2, 1), (3, 2), (3, 2), (3, 0), (2, 0), (1, 1), (1, 0), (2, 2), (3, 1), (3, 3), (3, 2), (2, 0), (0, 3), (3, 2), (1, 1), (1, 0), (3, 2), (1, 2), (1, 3), (1, 0), (0, 2), (2, 3), (0, 3), (3, 0), (0, 2), (1, 2), (2, 2), (2, 3), (0, 2), (0, 0), (2, 2), (3, 2), (3, 1), (2, 3), (1, 0), (1, 1), (0, 0), (0, 0), (3, 0), (3, 2), (0, 0), (3, 3), (2, 0), (3, 0), (0, 3), (3, 1), (3, 3), (1, 2), (3, 1), (0, 0), (0, 0), (2, 1), (1, 1), (3, 2), (2, 2), (0, 2), (0, 3), (3, 3), (0, 2), (1, 2), (0, 3), (0, 0), (2, 2), (0, 3), (0, 3), (2, 0), (2, 1), (2, 2), (1, 2), (1, 1), (3, 1), (1, 0), (0, 2), (1, 3), (0, 0), (0, 3), (0, 3), (3, 2), (0, 3), (3, 0), (3, 0), (1, 1), (1, 2), (3, 1), (2, 0), (3, 0), (3, 3)]
Shuffle output: [(3, [1, 3, 2, 2, 2, 0, 1, 3, 2, 2, 2, 0, 2, 1, 0, 2, 3, 0, 1, 3, 1, 2, 3, 1, 2, 0, 0, 1, 0, 3]), (2, [0, 1, 2, 1, 0, 2, 0, 3, 2, 3, 2, 3, 0, 1, 2, 2, 0, 1, 2, 0]), (0, [2

#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [52]:
# Конфигурация: разбиваем вектор на 2 сегмента
VECTOR_SEGMENTS = 2

def RECORDREADER(rows: int, cols: int):
    # Генерация матрицы M (row_idx, col_idx, value)
    matrix = []
    for r in range(rows):
        for c in range(cols):
            matrix.append((r, c, random.randint(1, 10)))

    # Генерация вектора V (index, value)
    vector = [(i, random.randint(1, 5)) for i in range(cols)]

    return matrix, vector

def MAP(matrix_cell: tuple, vector_segment: List[tuple]):
    # matrix_cell: (i, j, m_ij)
    # vector_segment: список пар (j, v_j) для конкретного диапазона j

    row_i, col_j, m_val = matrix_cell

    # Ищем соответствующее значение v_j в полученном сегменте
    for v_idx, v_val in vector_segment:
        if v_idx == col_j:
            # Вычисляем частичное произведение
            return (row_i, m_val * v_val)
    return None

def PARTITION(key: int, num_reducers: int):
    return key % num_reducers

def tetta(values: List[float]) -> float:
    # Суммируем все частичные произведения для строки i
    return sum(values)

def REDUCE(row_index: int, partial_products: List[float]):
    # Финальная агрегация для получения компоненты результирующего вектора
    res_value = tetta(partial_products)
    return (row_index, res_value)

def group_by_key(data):
    storage = {}
    for k, v in data:
        if v is not None:
            if k not in storage:
                storage[k] = []
            storage[k].append(v)
    return list(storage.items())


m_data, v_data = RECORDREADER(4, 4)

# Разделение вектора (имитация того, что Mapper видит только часть)
v_seg1 = v_data[:2]
v_seg2 = v_data[2:]

# Фаза Map: обработка разных частей матрицы с разными частями вектора
map_output = []
for cell in m_data:
    # Определяем, какой сегмент вектора нужен для данного столбца
    current_seg = v_seg1 if cell[1] < 2 else v_seg2
    res = MAP(cell, current_seg)
    if res:
        map_output.append(res)

# Фаза Shuffle
shuffled = group_by_key(map_output)

# Фаза Reduce
result_vector = [REDUCE(k, v) for k, v in shuffled]
result_vector.sort()

print("Resulting Vector (index, value):", result_vector)

Resulting Vector (index, value): [(0, 62), (1, 69), (2, 54), (3, 21)]


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [22]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [58]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  # solution code that yield(k2,v2) pairs

def REDUCE(key, values):
  (i, k) = key
  # solution code that yield(k3,v3) pairs

Проверьте своё решение

In [59]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

TypeError: 'NoneType' object is not iterable

In [ ]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [63]:
# Параметры размерности
DIM_I, DIM_J, DIM_K = 2, 3, 40

# Инициализация
mat_m = np.random.rand(DIM_I, DIM_J)
mat_n = np.random.rand(DIM_J, DIM_K)
reference_solution = np.matmul(mat_m, mat_n)

def RECORDREADER():
    # Потоковая генерация элементов матриц
    for r in range(mat_m.shape[0]):
        for c in range(mat_m.shape[1]):
            yield ((0, r, c), mat_m[r, c])
    for r in range(mat_n.shape[0]):
        for c in range(mat_n.shape[1]):
            yield ((1, r, c), mat_n[r, c])

def MAP_JOIN(coords, val):
    tag, i, j = coords
    # Реализация объединения по общему индексу
    if tag == 0:
        yield (j, (tag, i, val))
    else:
        yield (i, (tag, j, val))

def REDUCE_JOIN(join_key, values):
    # Разделение по источникам (M и N)
    part_m = [item for item in values if item[0] == 0]
    part_n = [item for item in values if item[0] == 1]

    for m_item in part_m:
        for n_item in part_n:
            # Выход: ((строка_M, столбец_N), произведение)
            yield ((m_item[1], n_item[1]), m_item[2] * n_item[2])

def MAP_SUM(idx_pair, prod):
    yield (idx_pair, prod)

def REDUCE_SUM(idx_pair, prod_list):
    # Финальная агрегация элементов
    sum_val = 0
    for p in prod_list:
        sum_val += p
    yield (idx_pair, sum_val)

def flatten(iterable):
    for sub in iterable:
        if isinstance(sub, tuple) and not any(isinstance(i, (list, tuple)) for i in sub):
            yield sub
        else:
            for item in sub:
                yield item

def groupbykey(iterable):
    buckets = {}
    for k, v in iterable:
        if k not in buckets:
            buckets[k] = []
        buckets[k].append(v)
    return buckets.items()

def MapReduce(READER, MAP_FUNC, REDUCE_FUNC):
    # Важно: используем flatten после MAP, так как MAP теперь генератор (yield)
    map_res = flatten(map(lambda x: MAP_FUNC(*x), READER()))
    grouped = groupbykey(map_res)
    return list(flatten(map(lambda x: REDUCE_FUNC(*x), grouped)))


# Этап 1: Соединение элементов
step1_result = MapReduce(RECORDREADER, MAP_JOIN, REDUCE_JOIN)

# Этап 2: Агрегация (сумма произведений)
def INTERMEDIATE_READER():
    return step1_result

final_pairs = MapReduce(INTERMEDIATE_READER, MAP_SUM, REDUCE_SUM)

# Сборка матрицы для проверки
def asmatrix(res_list):
    mat = np.zeros((DIM_I, DIM_K))
    for (r, c), v in res_list:
        mat[r, c] = v
    return mat

print("Результат верен:", np.allclose(reference_solution, asmatrix(final_pairs)))

Результат верен: True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [62]:
# Определение размерностей
DIM_I, DIM_J, DIM_K = 2, 3, 40
reducers = 2

# Инициализация матриц
m_matrix = np.random.rand(DIM_I, DIM_J)
n_matrix = np.random.rand(DIM_J, DIM_K)
reference_solution = np.matmul(m_matrix, n_matrix)

def flatten(content):
    for block in content:
        for unit in block:
            yield unit

def groupbykey(data_stream):
    cache = {}
    for (k, v) in data_stream:
        if k not in cache:
            cache[k] = []
        cache[k].append(v)
    return cache.items()

def PARTITIONER(key_obj):
    return hash(key_obj) % reducers

def groupbykey_distributed(partitions_list, hash_func):
    shuffled_nodes = [dict() for _ in range(reducers)]
    for chunk in partitions_list:
        for (key, val) in chunk:
            target_idx = hash_func(key)
            node = shuffled_nodes[target_idx]
            if key not in node:
                node[key] = []
            node[key].append(val)
    return [(idx, sorted(node.items())) for (idx, node) in enumerate(shuffled_nodes)]

def MapReduceDistributed(INPUT_SENDER, MAP_OP, REDUCE_OP, PART_FUNC=PARTITIONER, COMBINE_OP=None):
    # Фаза Map
    mapped_shards = map(lambda reader: flatten(map(lambda kv: MAP_OP(*kv), reader)), INPUT_SENDER())

    # Локальная агрегация
    if COMBINE_OP is not None:
        mapped_shards = map(lambda shard: flatten(map(lambda kv: COMBINE_OP(*kv), groupbykey(shard))), mapped_shards)

    # Фаза Shuffle
    network_transfer = groupbykey_distributed(mapped_shards, PART_FUNC)

    # Фаза Reduce
    output_stream = map(lambda node: (node[0], flatten(map(lambda group: REDUCE_OP(*group), node[1]))), network_transfer)

    return output_stream

def INPUTFORMAT():
    # Отдельный генератор для каждой матрицы
    m_elements = [((0, r, c), m_matrix[r, c]) for r in range(DIM_I) for c in range(DIM_J)]
    yield m_elements

    n_elements = [((1, r, c), n_matrix[r, c]) for r in range(DIM_J) for c in range(DIM_K)]
    yield n_elements

def MAP_PHASE_JOIN(meta, val):
    m_idx, r, c = meta
    # Ключ для Join — общий индекс j
    if m_idx == 0:
        yield (c, (m_idx, r, val))
    else:
        yield (r, (m_idx, c, val))

def REDUCE_PHASE_JOIN(shared_key, values):
    left = [v for v in values if v[0] == 0]
    right = [v for v in values if v[0] == 1]
    for a in left:
        for b in right:
            # Выход: ((i, k), m_ij * n_jk)
            yield ((a[1], b[1]), a[2] * b[2])

def MAP_PHASE_MUL(coords, product):
    yield (coords, product)

def REDUCE_PHASE_MUL(cell_idx, products):
    # Суммирование всех произведений для ячейки (i, k)
    yield (cell_idx, sum(products))


# 1. Этап Join
distributed_join = MapReduceDistributed(INPUTFORMAT, MAP_PHASE_JOIN, REDUCE_PHASE_JOIN)
join_results = [(p_id, list(p_data)) for (p_id, p_data) in distributed_join]

def GET_JOINED_DATA():
    for partition in join_results:
        yield partition[1]

# 2. Этап Сложения
distributed_mul = MapReduceDistributed(GET_JOINED_DATA, MAP_PHASE_MUL, REDUCE_PHASE_MUL)
final_parts = [(p_id, list(p_data)) for (p_id, p_data) in distributed_mul]

# Сборка финального списка
flat_solution = [record for p in final_parts for record in p[1]]

def asmatrix(data_list):
    # Определение границ матрицы
    max_r = max(x[0][0] for x in data_list) + 1
    max_c = max(x[0][1] for x in data_list) + 1
    matrix_out = np.zeros((int(max_r), int(max_c)))
    for (coords, val) in data_list:
        matrix_out[coords[0], coords[1]] = val
    return matrix_out

# Валидация
print("Check passed:", np.allclose(reference_solution, asmatrix(flat_solution)))

Check passed: True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [61]:
# Инициализация
I, J, K = 2, 3, 40
reducers = 2
maps = 3  # Количество ридеров на каждую матрицу

m_mat = np.random.rand(I, J)
n_mat = np.random.rand(J, K)
reference_solution = np.matmul(m_mat, n_mat)

def flatten(content):
    for block in content:
        for item in block:
            yield item

def groupbykey(stream):
    res = {}
    for k, v in stream:
        if k not in res: res[k] = []
        res[k].append(v)
    return res.items()

def PARTITIONER(key):
    return hash(key) % reducers

def groupbykey_distributed(map_outputs, partition_func):
    shards = [dict() for _ in range(reducers)]
    for chunk in map_outputs:
        for k, v in chunk:
            p_idx = partition_func(k)
            target = shards[p_idx]
            if k not in target: target[k] = []
            target[k].append(v)
    return [(i, sorted(s.items())) for i, s in enumerate(shards)]

def MapReduceDistributed(INPUT_GEN, MAP_F, REDUCE_F, PART_F=PARTITIONER, COMBINER=None):
    # Применяем MAP к каждому сплиту от каждого RECORDREADER
    mapped = map(lambda split: flatten(map(lambda kv: MAP_F(*kv), split)), INPUT_GEN())

    if COMBINER:
        mapped = map(lambda m: flatten(map(lambda kv: COMBINER(*kv), groupbykey(m))), mapped)

    # Shuffle & Sort
    shuffled = groupbykey_distributed(mapped, PART_F)

    # Reduce
    results = map(lambda node: (node[0], flatten(map(lambda g: REDUCE_F(*g), node[1]))), shuffled)
    return results

def INPUTFORMAT():
    """Генерирует несколько сплитов для каждой матрицы"""
    def create_splits(data, n):
        random.shuffle(data) # Перемешивание подтверждает работу со случайными подмножествами
        size = int(np.ceil(len(data) / n))
        for i in range(0, len(data), size):
            yield data[i:i + size]

    m_data = [((0, r, c), m_mat[r, c]) for r in range(I) for c in range(J)]
    n_data = [((1, r, c), n_mat[r, c]) for r in range(J) for c in range(K)]

    import random
    yield from create_splits(m_data, maps)
    yield from create_splits(n_data, maps)

def MAP_JOIN(meta, val):
    tag, r, c = meta
    if tag == 0: yield (c, (tag, r, val)) # j - ключ
    else: yield (r, (tag, c, val))        # j - ключ

def REDUCE_JOIN(j, values):
    m_side = [v for v in values if v[0] == 0]
    n_side = [v for v in values if v[0] == 1]
    for m in m_side:
        for n in n_side:
            yield ((m[1], n[1]), m[2] * n[2])

def MAP_SUM(coords, val):
    yield (coords, val)

def REDUCE_SUM(coords, vals):
    yield (coords, sum(vals))


# 1. Join
res_join = MapReduceDistributed(INPUTFORMAT, MAP_JOIN, REDUCE_JOIN)
joined_data = [list(p[1]) for p in res_join]

def GET_JOINED():
    for partition in joined_data:
        yield partition

# 2. Sum
res_sum = MapReduceDistributed(GET_JOINED, MAP_SUM, REDUCE_SUM)
final_list = [record for p in res_sum for record in p[1]]

# Сборка и проверка
def asmatrix(data):
    res = np.zeros((I, K))
    for (coords, val) in data:
        res[coords[0], coords[1]] = val
    return res

calculated = asmatrix(final_list)
print(f"Результат корректен: {np.allclose(reference_solution, calculated)}")

Результат корректен: True
